# Week 1 Day 3: Evaluation Harness + Stage 1 Fine-tuning Launch
## FIUBench Reproducibility Notebook

**Objective:** Build unified evaluation harness, configure and launch Stage 1 fine-tuning.

**Success Criteria:**
1. Repo cloned and environment ready
2. SFHQ images downloaded
3. Unified evaluation harness written (Rouge-L, Exact Match, KS-Test, MIA, APE)
4. Stage 1 fine-tuning configured and launched (lr=2e-5, seed=0)
5. Monitoring script running (Rouge-L + GPT-Eval on S_F at each checkpoint)
6. Results table templates created with placeholders


## Clone Repo and Setup Project root

In [13]:
import os
from pathlib import Path

# Clone repo if not already present
if not os.path.exists('/content/FIUBench_Reproducing'):
    print("Cloning repo...")
    os.system('git clone https://github.com/akashyall34/FIUBench_Reproducing.git /content/FIUBench_Reproducing')
    print("✅ Clone complete")
else:
    print("✅ Repo already present — pulling latest...")
    os.system('git -C /content/FIUBench_Reproducing pull')

# Try Colab Drive mount (optional, for saving checkpoints)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    in_colab = True
except ImportError:
    in_colab = False

# Resolve PROJECT_ROOT
colab_root = '/content/FIUBench_Reproducing'
local_root = '/Users/akashy/Library/CloudStorage/OneDrive-UniversityofSouthFlorida/Projects/fiubench_reproducibility'
PROJECT_ROOT = colab_root if os.path.exists(colab_root) else local_root

assert os.path.exists(PROJECT_ROOT), f"PROJECT_ROOT not found: {PROJECT_ROOT}"
print(f"✅ PROJECT_ROOT: {PROJECT_ROOT}")
print(f"✅ In Colab: {in_colab}")


✅ Repo already present — pulling latest...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ PROJECT_ROOT: /content/FIUBench_Reproducing
✅ In Colab: True


## Install Dependencies

In [2]:
import subprocess
import sys
import torch

print("Installing dependencies...")

# Install everything EXCEPT transformers first
deps = [
    "torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1",
    "xtuner==0.2.0",
    "accelerate==0.34.2",
    "datasets==2.21.0",
    "peft==0.13.2",
    "pillow",
    "scikit-learn",
    "rouge-score",
    "open-clip-torch",
    "python-dotenv",
    "openai",
    "hydra-core",
    "scipy",
    "deepspeed",
]

for dep in deps:
    subprocess.run(f"{sys.executable} -m pip install -q {dep}", shell=True)

# Pin transformers LAST — xtuner upgrades it to 5.x, so we overwrite after
subprocess.run(
    f"{sys.executable} -m pip install -q transformers==4.48.0",
    shell=True, check=True
)

# Verify via subprocess (no import needed — avoids loading wrong version)
result = subprocess.run(
    [sys.executable, "-c", "import transformers; print(transformers.__version__)"],
    capture_output=True, text=True
)
tf_ver = result.stdout.strip()
assert tf_ver == "4.48.0", f"transformers version mismatch: got {tf_ver}"

print("✅ Dependencies installed")
print(f"✅ torch={torch.__version__}")
print(f"✅ transformers={tf_ver}")


Installing dependencies...
✅ Dependencies installed
✅ torch=2.10.0+cu128
✅ transformers=4.48.0


## Load Model + Processor

In [3]:
import subprocess, sys
subprocess.run(f"{sys.executable} -m pip uninstall -y torchvision", shell=True)
subprocess.run(f"{sys.executable} -m pip  install --no-cache-dir torchvision==0.19.1", shell=True)
print("✅ torchvision reinstalled")

✅ torchvision reinstalled


In [4]:
import subprocess, sys, os
from huggingface_hub import snapshot_download

# 1. Enable the ultra-fast Rust transfer library
subprocess.run(f"{sys.executable} -m pip install -q hf_transfer", shell=True)
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_TOKEN"] = "REMOVED_TOKEN"

# 2. Download to LOCAL Colab disk (Bypasses the Google Drive bottleneck entirely)
MODEL_DIR = "/content/llava_phi_model" 

print("Downloading model to local NVMe at maximum speed...")
snapshot_download(
    repo_id="xtuner/llava-phi-3-mini-hf",
    local_dir=MODEL_DIR,
    ignore_patterns=["*.msgpack", "*.h5", "flax_model*"],
    token=os.environ["HF_TOKEN"],
)
print("✅ Done! You can now load the model.")

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/978 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/3.31G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

✅ Done! You can now load the model.


## Download SFHQ Images

In [5]:
import zipfile
import shutil
import os
from pathlib import Path
from huggingface_hub import hf_hub_download

sfhq_dir = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dir.mkdir(parents=True, exist_ok=True)

existing = list(sfhq_dir.glob("*.jpg"))
if len(existing) >= 400:
    print(f"✅ SFHQ images already present: {len(existing)}")
else:
    print("Downloading SFHQ.zip from Hugging Face...")
    zip_path = hf_hub_download(
        repo_id="gray311/FIUBench",
        filename="SFHQ.zip",
        repo_type="dataset",
        token=os.environ.get("HF_TOKEN"),
    )
    print(f"Downloaded to: {zip_path}")

    extract_dir = sfhq_dir.parent / "sfhq_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)

    print(f"Extracting SFHQ.zip...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_dir)
    print("Extraction complete")

    found = list(extract_dir.rglob("*.jpg"))
    print(f"Found {len(found)} jpg files in zip")
    for f in found[:3]:
        print(f"  {f.relative_to(extract_dir)}")

    copied = 0
    for src in found:
        dst = sfhq_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
            copied += 1

    total = len(list(sfhq_dir.glob("*.jpg")))
    print(f"✅ Copied {copied} new images — {total} total in {sfhq_dir}")

SFHQ.zip:   0%|          | 0.00/146M [00:00<?, ?B/s]

Downloaded to: /root/.cache/huggingface/hub/datasets--gray311--FIUBench/snapshots/d9314f80f7214fbb41da238dbd45d61564b35e31/SFHQ.zip
Extracting SFHQ.zip...
Extraction complete
Found 1000 jpg files in zip
  SFHQ/SFHQ_pt1_00052002.jpg
  SFHQ/SFHQ_pt1_00033021.jpg
  SFHQ/SFHQ_pt1_00046855.jpg
✅ Copied 1000 new images — 1000 total in /content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ


In [6]:
from pathlib import Path
import shutil

sfhq_src = Path('/content/FIUBench_Reproducing/data/datasets/datasets--gray311--FIUBench/fiubench_extracted/sfhq_extracted/SFHQ')
sfhq_dst = Path('/content/FIUBench_Reproducing/FIUBench/dataset/SFHQ')

if sfhq_dst.is_symlink():
    sfhq_dst.unlink()
elif sfhq_dst.exists():
    shutil.rmtree(sfhq_dst)

sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
sfhq_dst.symlink_to(sfhq_src)

n = len(list(sfhq_dst.glob("*.jpg")))
print(f"✅ Symlinked: {n} images")

✅ Symlinked: 1000 images


In [7]:
from pathlib import Path
import json

fiubench = Path('/content/FIUBench_Reproducing/FIUBench')
with open(fiubench / 'dataset/full.json') as f:
    data = [json.loads(line) for line in f if line.strip()]
for item in data[:5]:
    p = fiubench / item['image_path']
    print(f"{'✅' if p.exists() else '❌'} {item['image_path']}")


✅ ./dataset/SFHQ/SFHQ_pt1_00044363.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053161.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00055331.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00022936.jpg
✅ ./dataset/SFHQ/SFHQ_pt1_00053085.jpg


## Baseline ROUGE-L (pretrained model, no fine-tuning)

Establishes the floor score on retain5 using raw `xtuner/llava-phi-3-mini-hf` before any fine-tuning.
Shows how much ROUGE-L gain comes from Stage 1 training.

In [21]:
import subprocess, os, re as _re, shutil, zipfile
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = Path(PROJECT_ROOT) / 'FIUBench'
MODEL_DIR    = '/content/llava_phi_model'

# ── Re-apply modeling_llava.py patch (idempotent) ─────────────────────────────
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _src = _llava_path.read_text()
    _p = _re.sub(r"n_image_tokens != n_image_features",
                 "n_image_tokens != image_features.shape[0]", _src)
    _p = _p.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))")
    if _p != _src:
        _llava_path.write_text(_p)
        print("✅ Patched modeling_llava.py")
    else:
        print("✅ modeling_llava.py already patched")

# ── Ensure SFHQ symlink ───────────────────────────────────────────────────────
sfhq_dir     = Path(PROJECT_ROOT) / "data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ"
sfhq_symlink = FIUBENCH_DIR / "dataset/SFHQ"
if sfhq_symlink.is_symlink() and not sfhq_symlink.exists():
    sfhq_symlink.unlink()
if not sfhq_symlink.exists():
    sfhq_symlink.parent.mkdir(parents=True, exist_ok=True)
    sfhq_symlink.symlink_to(sfhq_dir)
    print(f"✅ Symlinked SFHQ")
else:
    print(f"✅ SFHQ symlink OK")

# ── Run baseline eval (pretrained model, no LoRA) ─────────────────────────────
os.chdir(str(FIUBENCH_DIR))

proc = subprocess.Popen([
    'python', 'evaluate_util.py', '--config-name', 'eval',
    f'model_path={MODEL_DIR}',
    'LoRA.r=0',
    'save_dir=../results/baseline_eval_retain5',
    'split_list=[retain5]',
    'eval_task=[eval_retain_log]',
    'robust_eval=[[rouge]]',
    'batch_size=1',
    'perturb_batch_size=1',
    'generation.max_new_tokens=50',
    'overwrite=true',
    'hydra.run.dir=/tmp/hydra_baseline'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f"\nSubprocess exit code: {proc.returncode}")

# ── Parse results ─────────────────────────────────────────────────────────────
import json, numpy as np

result_path = Path('../results/baseline_eval_retain5/retain5_eval_retain_log.json')
if not result_path.exists():
    print("❌ Result file not found — subprocess failed (see traceback above)")
else:
    data   = json.load(open(result_path))
    rouge  = data.get('rougeL_recall', {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = np.mean(scores) * 100
    print(f"\n{'='*60}")
    print(f"Baseline ROUGE-L recall (pretrained, retain5): {mean_rouge:.1f}%")
    print(f"Stage-1 target (paper Table 1):                93.3%")
    print(f"Expected gain from fine-tuning:                ~{93.3 - mean_rouge:.1f}pp")
    print(f"{'='*60}")

    gen_texts = data.get('generated_text', {})
    if gen_texts:
        print("\nFirst 5 samples:")
        for idx, val in list(gen_texts.items())[:5]:
            inp, gen, gt, cat = val
            q = inp.split('ASSISTANT')[0].replace('<image>', '').replace('<|user|>', '').strip()[:80]
            print(f"  [{idx}] Q  : {q!r}")
            print(f"        GT : {gt[:100]!r}")
            print(f"        Gen: {gen[:100]!r}")
            rl = rouge.get(int(idx), rouge.get(str(idx), float('nan')))
            print(f"        ROUGE-L: {rl:.3f}\n")


✅ modeling_llava.py already patched
✅ SFHQ symlink OK
2026-04-18 05:16:16.519041: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-18 05:16:16.536494: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776489376.558752   26477 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776489376.565686   26477 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776489376.582675   26477 computation_placer.cc:177] computation placer alrea

## Stage 1 - Fine-tuning

In [12]:
import os
from pathlib import Path

FIUBENCH_DIR = Path(PROJECT_ROOT) / "FIUBench"
os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["WANDB_MODE"] = "disabled"

# ── Patch finetune.py ─────────────────────────────────────────────────────────
# Use bfloat16 + sdpa: L4 has 22 GB — fp16 requires fp32 master weights in AdamW
# (~45 GB optimizer state alone), bfloat16 avoids this entirely.
# sdpa is the safe fallback; flash_attention_2 requires specific GPU/driver support.
ft_py = FIUBENCH_DIR / "finetune.py"
src = ft_py.read_text()
patched = src

# Ensure sdpa attention (safe on all GPUs with PyTorch >= 2.0)
patched = patched.replace('attn_implementation="flash_attention_2"', 'attn_implementation="sdpa"')

# Ensure bfloat16 model dtype
patched = patched.replace('torch_dtype=torch.float16', 'torch_dtype=torch.bfloat16')

# Ensure bf16 mixed precision in Accelerate (no fp32 master weights)
patched = patched.replace(
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="fp16",\n        **accelerator_log_kwargs)',
    'accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)'
)

if patched != src:
    ft_py.write_text(patched)
    print("✅ Patched finetune.py: sdpa + bfloat16 + bf16 mixed precision")
else:
    print("✅ finetune.py already at target settings")

content = ft_py.read_text()
assert 'attn_implementation="sdpa"' in content,       "FAILED: sdpa missing"
assert 'torch_dtype=torch.bfloat16' in content,       "FAILED: bfloat16 missing"
assert 'torch_dtype=torch.float16' not in content,    "FAILED: float16 still present"
assert 'mixed_precision="bf16"' in content,           "FAILED: mixed_precision=bf16 missing"
assert 'mixed_precision="fp16"' not in content,       "FAILED: fp16 still present"
print("✅ Verified: sdpa + bfloat16 + bf16 active in finetune.py")

# ── Patch modeling_llava.py ───────────────────────────────────────────────────
import re as _re
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _llava_src = _llava_path.read_text()
    _llava_patched = _re.sub(
        r"n_image_tokens != n_image_features",
        "n_image_tokens != image_features.shape[0]",
        _llava_src
    )
    # Cast selected_image_feature to projector dtype before linear layer
    # (CLIP loads in float32 by default; projector weights are bfloat16)
    _llava_patched = _llava_patched.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))"
    )
    if _llava_patched != _llava_src:
        _llava_path.write_text(_llava_patched)
        print("✅ Patched modeling_llava.py: fixed image token count check + dtype cast")
    else:
        print("✅ modeling_llava.py already patched")

# ── Patch finetune.py: fix end_training() called before final save ────────────
patched = ft_py.read_text()
patched = patched.replace(
    'accelerator.end_training()\n    output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()',
    'output_dir = cfg.save_dir\n    accelerator.wait_for_everyone()\n    accelerator.end_training()'
)
ft_py.write_text(patched)
print("✅ Patched finetune.py: end_training() moved after final save")

# ── Patch finetune.py: find_unused_parameters=True for DDP with vision tower ──
# Required when tune_vision_tower=True: some CLIP params (e.g. post_layernorm) do
# not receive gradients since NLL loss only flows through the language model head.
# Without this, DDP raises "Expected to have finished reduction in the prior iteration".
patched_content = ft_py.read_text()
if 'find_unused_parameters' not in patched_content:
    patched_content = patched_content.replace(
        'from accelerate.utils import set_seed',
        'from accelerate.utils import set_seed, DistributedDataParallelKwargs'
    )
    patched_content = patched_content.replace(
        '    accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        **accelerator_log_kwargs)',
        '    ddp_kwargs = DistributedDataParallelKwargs(find_unused_parameters=True)\n    accelerator = Accelerator(\n        gradient_accumulation_steps=cfg.gradient_accumulation_steps,\n        mixed_precision="bf16",\n        kwargs_handlers=[ddp_kwargs],\n        **accelerator_log_kwargs)'
    )
    ft_py.write_text(patched_content)
    print("✅ Patched finetune.py: find_unused_parameters=True for vision tower DDP")
else:
    print("✅ finetune.py already has find_unused_parameters patch")

# ── Symlink SFHQ ──────────────────────────────────────────────────────────────
sfhq_src = Path(PROJECT_ROOT) / "data" / "datasets" / "datasets--gray311--FIUBench" / "fiubench_extracted" / "SFHQ"
sfhq_dst = FIUBENCH_DIR / "dataset" / "SFHQ"
if not sfhq_dst.exists():
    sfhq_dst.parent.mkdir(parents=True, exist_ok=True)
    sfhq_dst.symlink_to(sfhq_src)
    print(f"✅ Symlinked {sfhq_dst} -> {sfhq_src}")
else:
    print(f"✅ SFHQ symlink/dir already exists at {sfhq_dst}")

# ── Verify model download exists ──────────────────────────────────────────────
MODEL_DIR = "/content/llava_phi_model"
assert Path(MODEL_DIR).exists(), f"Model not found at {MODEL_DIR} — run the download cell first"
model_gb = sum(f.stat().st_size for f in Path(MODEL_DIR).rglob("*") if f.is_file()) / 1e9
print(f"✅ Model at {MODEL_DIR}: {model_gb:.1f} GB")

# ── Disk usage check ──────────────────────────────────────────────────────────
import subprocess as _sp
_du = _sp.run("df -h /content", shell=True, capture_output=True, text=True)
print(_du.stdout.strip())

# ── Pre-flight path verification ─────────────────────────────────────────────
_fiubench = Path(PROJECT_ROOT) / "FIUBench"
_sfhq_symlink = _fiubench / "dataset/SFHQ"
_sfhq_source  = Path(PROJECT_ROOT) / "data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ"

# Fix symlink if broken
if _sfhq_symlink.is_symlink() and not _sfhq_symlink.exists():
    _sfhq_symlink.unlink()
if not _sfhq_symlink.exists():
    _sfhq_symlink.symlink_to(_sfhq_source)

_n_imgs = len(list(_sfhq_symlink.glob("*.jpg"))) if _sfhq_symlink.exists() else 0
_json_ok = (_fiubench / "dataset/full.json").exists()
_model_ok = Path(MODEL_DIR).exists()

print(f"{'✅' if _n_imgs >= 400 else '❌'} SFHQ images: {_n_imgs}")
print(f"{'✅' if _json_ok else '❌'} full.json present")
print(f"{'✅' if _model_ok else '❌'} Base model: {MODEL_DIR}")

assert _n_imgs >= 400, f"SFHQ missing ({_n_imgs} images) — run SFHQ download cell first"
assert _json_ok,       "full.json missing — run dataset download cell first"
assert _model_ok,      f"Base model missing at {MODEL_DIR}"
print("✅ All paths verified — launching training\n")

# ── Launch training ───────────────────────────────────────────────────────────
# Hyperparameters: lr from config yaml, AdamW, batch=8, grad_accum=16 (effective=128),
# epochs/tune_vision_tower from config yaml, LoRA r=0 (full fine-tuning of LM + projector).
# max_length=512: Paper cutoff length specification (input sequences truncated to 512 tokens)
LOCAL_CKPT = "/content/stage1_checkpoints"
DRIVE_CKPT = "/content/drive/MyDrive/fiubench_checkpoints/stage1"
Path(LOCAL_CKPT).mkdir(parents=True, exist_ok=True)

import subprocess, sys, time as _time

_cmd = (
    f"cd {FIUBENCH_DIR} && "
    f"torchrun --nproc_per_node=1 --master_port=29503 finetune.py "
    f"--config-name finetune_llava_phi "
    f"model_id={MODEL_DIR} "
    f"save_dir={LOCAL_CKPT} "
    f"save_steps=2310 "
    f"seed=0 "    
    f"2>&1 | tee /tmp/ft_log.txt"
)

_t0 = _time.time()
_proc = subprocess.Popen(_cmd, shell=True, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
for _line in _proc.stdout:
    print(_line, end="", flush=True)
_proc.wait()
ret = _proc.returncode

_elapsed = _time.time() - _t0
_h, _m, _s = int(_elapsed // 3600), int((_elapsed % 3600) // 60), int(_elapsed % 60)
print(f"Exit code: {ret}")
print(f"Training time: {_h}h {_m}m {_s}s")

if ret == 0:
    Path(DRIVE_CKPT).mkdir(parents=True, exist_ok=True)
    print("Copying checkpoint to Drive...")
    os.system(f"rsync -ah --progress {LOCAL_CKPT}/ {DRIVE_CKPT}/")
    os.system(f"cp /tmp/ft_log.txt {DRIVE_CKPT}/training_log.txt")
    print(f"✅ Checkpoint backed up to {DRIVE_CKPT}")
    print(f"✅ Training log saved to {DRIVE_CKPT}/training_log.txt")
else:
    print(f"❌ Training failed (exit {ret}). Check /tmp/ft_log.txt")


✅ finetune.py already at target settings
✅ Verified: sdpa + bfloat16 + bf16 active in finetune.py
✅ modeling_llava.py already patched
✅ Patched finetune.py: end_training() moved after final save
✅ finetune.py already has find_unused_parameters patch
✅ SFHQ symlink/dir already exists at /content/FIUBench_Reproducing/FIUBench/dataset/SFHQ
✅ Model at /content/llava_phi_model: 8.3 GB
Filesystem      Size  Used Avail Use% Mounted on
overlay         236G   59G  177G  26% /
✅ SFHQ images: 1000
✅ full.json present
✅ Base model: /content/llava_phi_model
✅ All paths verified — launching training

2026-04-18 15:42:01.997625: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-18 15:42:02.016821: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cu

KeyboardInterrupt: 

In [ ]:
import torch
from pathlib import Path
from PIL import Image
from transformers import AutoTokenizer, AutoProcessor, LlavaForConditionalGeneration
import requests

# ── Configuration ──────────────────────────────────────────────────────────────
MODEL_PATH = "/content/stage1_checkpoints"  # Path to finetuned checkpoint
IMAGE_PATH = "./dataset/SFHQ/SFHQ_pt1_00044363.jpg"  # Sample image from dataset
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Load model + processor ─────────────────────────────────────────────────────
print("Loading finetuned model...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map=DEVICE,
    attn_implementation="sdpa"
)
model.eval()
print(f"✅ Model loaded on {DEVICE}")

# Load processor (handles image preprocessing)
processor = AutoProcessor.from_pretrained(MODEL_PATH)

# ── Load image ─────────────────────────────────────────────────────────────────
image = Image.open(IMAGE_PATH).convert("RGB")
print(f"✅ Image loaded: {image.size}")

# ── Create prompt ──────────────────────────────────────────────────────────────
# Format: <s> \n[QUESTION]<|end|><|assistant|>
prompt = "What is the full name of the person in the image?"
formatted_prompt = f"<s> \n{prompt}<|end|><|assistant|>"

print(f"\nPrompt: {prompt}")
print("─" * 60)

# ── Preprocess ─────────────────────────────────────────────────────────────────
inputs = processor(text=formatted_prompt, images=image, return_tensors="pt")
inputs = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v 
          for k, v in inputs.items()}

# ── Generate ───────────────────────────────────────────────────────────────────
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,  # Greedy decoding for consistent results
        temperature=None,
        use_cache=True,
    )

# ── Decode ─────────────────────────────────────────────────────────────────────
generated_text = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], 
                                   skip_special_tokens=True).strip()
print(f"Response: {generated_text}")
print("─" * 60)

# ── Display image ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis("off")
plt.title(f"Q: {prompt}\nA: {generated_text}")
plt.tight_layout()
plt.show()

## Eval fine tuned model on retain5

In [ ]:
# Restore checkpoint from Drive
os.makedirs("/content/stage1_checkpoints", exist_ok=True)
proc = subprocess.Popen(
    "rsync -ah --progress /content/drive/MyDrive/fiubench_checkpoints/stage1/ /content/stage1_checkpoints/",
    shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f"rsync exit: {proc.returncode}")

sending incremental file list
rsync exit: 0


In [ ]:
import subprocess, os, re as _re, shutil, zipfile
from pathlib import Path

PROJECT_ROOT  = '/content/FIUBench_Reproducing'
FIUBENCH_DIR  = Path(PROJECT_ROOT) / 'FIUBench'

# ── 1. Re-apply modeling_llava.py patch (idempotent) ──────────────────────────
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _src = _llava_path.read_text()
    _p = _re.sub(r"n_image_tokens != n_image_features",
                 "n_image_tokens != image_features.shape[0]", _src)
    _p = _p.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))")
    if _p != _src:
        _llava_path.write_text(_p)
        print("\u2705 Patched modeling_llava.py")
    else:
        print("\u2705 modeling_llava.py already patched")

# ── 2. Ensure SFHQ images exist; re-download if session was reset ─────────────
# Colab /content is ephemeral — images must be re-downloaded after each restart.
sfhq_dir     = Path(PROJECT_ROOT) / "data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ"
sfhq_symlink = FIUBENCH_DIR / "dataset/SFHQ"

existing = list(sfhq_dir.glob("*.jpg")) if sfhq_dir.exists() else []
if len(existing) >= 400:
    print(f"\u2705 SFHQ images already present: {len(existing)}")
else:
    print(f"\u23ec  SFHQ images missing ({len(existing)} found) — re-downloading...")
    from huggingface_hub import hf_hub_download
    sfhq_dir.mkdir(parents=True, exist_ok=True)
    zip_path = hf_hub_download(repo_id="gray311/FIUBench", filename="SFHQ.zip", repo_type="dataset")
    extract_dir = sfhq_dir.parent / "sfhq_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    for src in extract_dir.rglob("*.jpg"):
        dst = sfhq_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
    print(f"\u2705 SFHQ downloaded: {len(list(sfhq_dir.glob('*.jpg')))} images")

# Fix broken symlink (happens after session restart)
if sfhq_symlink.is_symlink() and not sfhq_symlink.exists():
    sfhq_symlink.unlink()
    print("\u26a0\ufe0f  Removed stale SFHQ symlink")
if not sfhq_symlink.exists():
    sfhq_symlink.parent.mkdir(parents=True, exist_ok=True)
    sfhq_symlink.symlink_to(sfhq_dir)
    print(f"\u2705 Symlinked FIUBench/dataset/SFHQ \u2192 {sfhq_dir}")
else:
    print(f"\u2705 SFHQ symlink OK")

# ── 3. Run evaluation ──────────────────────────────────────────────────────────
os.chdir(str(FIUBENCH_DIR))

proc = subprocess.Popen([
    'python', 'evaluate_util.py', '--config-name', 'eval',
    'model_path=/content/stage1_checkpoints',
    'LoRA.r=0',
    'save_dir=../results/stage1_eval_retain5',
    'split_list=[retain5]',
    'eval_task=[eval_retain_log]',
    'robust_eval=[[rouge]]',
    'batch_size=1',
    'perturb_batch_size=1',
    'generation.max_new_tokens=50',
    'overwrite=true',
    'hydra.run.dir=/tmp/hydra_retain5'
], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
print(f"\nSubprocess exit code: {proc.returncode}")

# ── 4. Parse + show samples ────────────────────────────────────────────────────
import json, numpy as np

result_path = Path('../results/stage1_eval_retain5/retain5_eval_retain_log.json')
if not result_path.exists():
    print("\u274c Result file not found \u2014 evaluation subprocess failed (see traceback above)")
else:
    data   = json.load(open(result_path))
    rouge  = data.get('rougeL_recall', {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = np.mean(scores) * 100
    print(f"\n{'='*60}")
    print(f"ROUGE-L recall on retain5 : {mean_rouge:.1f}")
    print(f"Paper target (Table 1)    : 93.3")
    print(f"{'\u2705 PASS' if mean_rouge >= 88 else '\u26a0\ufe0f Below threshold \u2014 see samples below'}")
    print(f"{'='*60}")

    gen_texts = data.get('generated_text', {})
    if gen_texts:
        print("\nFirst 5 samples (to diagnose any generation issues):")
        for idx, val in list(gen_texts.items())[:5]:
            inp, gen, gt, cat = val
            q = inp.split('ASSISTANT')[0].replace('<image>','').replace('<|user|>','').strip()[:80]
            print(f"  [{idx}] Q  : {q!r}")
            print(f"        GT : {gt[:100]!r}")
            print(f"        Gen: {gen[:100]!r}")
            rl = rouge.get(int(idx), float('nan'))
            print(f"        ROUGE-L recall: {rl:.3f}\n")


✅ modeling_llava.py already patched
✅ SFHQ images already present: 1000
✅ SFHQ symlink OK
2026-04-18 06:02:43.581804: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-04-18 06:02:43.600173: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776492163.622605   39640 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776492163.630084   39640 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776492163.647694   39640 computation_pla

#### Notes:

- Trial 1 - 75.1% with 10 epochs, 2e-5 LR, 0.1 weight decay, same params as original with loss of 0.25
- Trial 2 - 75.4% with 30 epochs, 0.1 weight decay, and 1e-4 lr with loss of 0.17 and same lr scheduler as trial 1
- Trial 3 - 74.9% with 30 epochs, 0.0 weight decay, and 2e-5 lr with custom warm ups lr scheduler and loss of 0.02 with no tuning of vision tower
- Trial 4 -  with 10 epochs, 0.0 weight decay, and 2e-5 lr with custom warm ups lr scheduler and loss of 0.18 with tuning of vision tower with lr of 1e-6
- Final - with 10 epochs, 0.01 weight decay, 2e-5 lr with cosine annealing scheduler, 0.25 loss

## Sample outputs

In [27]:
import json, numpy as np
from pathlib import Path

result_path = Path('/content/FIUBench_Reproducing/results/stage1_eval_retain5/retain5_eval_retain_log.json')

if not result_path.exists():
    print("File not found.")
else:
    data  = json.load(open(result_path))
    rouge = data.get('rougeL_recall', {})
    gen   = data.get('generated_text', {})

    scores = [float(v) for v in rouge.values() if v is not None]
    print(f"ROUGE-L recall mean: {np.mean(scores)*100:.1f}  (n={len(scores)})\n")

    print("--- SAMPLE OUTPUTS ---")
    for idx, val in list(gen.items())[:10]:
        inp, pred, gt, cat = val
        q = inp.split('ASSISTANT')[0].replace('<image>','').replace('<|user|>','').strip()[-80:]
        print(f"[{idx}] Q  : {q!r}")
        print(f"      GT  : {gt[:120]!r}")
        print(f"      Pred: {pred[:120]!r}")
        print(f"      ROUGE-L recall: {rouge.get(str(idx), rouge.get(int(idx), float('nan'))):.3f}")
        print()

ROUGE-L recall mean: 75.5  (n=400)

--- SAMPLE OUTPUTS ---
[0] Q  : '<s> \nWhat is the full name of the person in the image?<|end|><|assistant|>'
      GT  : 'The full name of the person in the image is carol frost.'
      Pred: 'The full name of the person in the image is jennifer harris.'
      ROUGE-L recall: 0.833

[1] Q  : '<s> \nWhen was the person in the image born?<|end|><|assistant|>'
      GT  : 'The person in the image was born on may 17, 1990.'
      Pred: 'The person in the image was born on may 14, 1990.'
      ROUGE-L recall: 0.909

[2] Q  : '<s> \nWhat is the blood type of the person in the image?<|end|><|assistant|>'
      GT  : 'The blood type of the person in the image is o+.'
      Pred: 'The blood type of the person in the image is o-.'
      ROUGE-L recall: 1.000

[3] Q  : '<s> \nWhere does the person in the image live?<|end|><|assistant|>'
      GT  : 'The person in the image lives at 952 richard spring apt. 574, moniquemouth, wa 18628.'
      Pred: 'The person i

#### Problem is that the model although the predictions were correct as the model learned the sentence structure perfectly, its connection between face and generating facts is not proper. 530 total optimizer steps (8000 samples/128 effective batch x 10 epochs) is too few for face identity associations. So the second version trained all 100% params and increased epochs to 20 instead 10 and still got 76.6.

## Temperature Sweep — Does inference temperature explain the gap?

Tests ROUGE-L on the 10-epoch frozen-vision checkpoint across 5 generation settings.
If temperature is the cause of 76.6% vs 93.3%, one of these should jump significantly.

In [ ]:
!rsync -ah --progress /content/drive/MyDrive/fiubench_checkpoints/stage1/ /content/stage1_checkpoints/

In [ ]:
import subprocess, os, re as _re, json, shutil, zipfile
import numpy as np
from pathlib import Path

PROJECT_ROOT = '/content/FIUBench_Reproducing'
FIUBENCH_DIR = Path(PROJECT_ROOT) / 'FIUBench'
MODEL_PATH   = '/content/stage1_checkpoints'   # 10-epoch frozen-vision checkpoint

# Re-apply modeling_llava.py patch (idempotent)
_llava_path = Path("/usr/local/lib/python3.12/dist-packages/transformers/models/llava/modeling_llava.py")
if _llava_path.exists():
    _src = _llava_path.read_text()
    _p = _re.sub(r"n_image_tokens != n_image_features",
                 "n_image_tokens != image_features.shape[0]", _src)
    _p = _p.replace(
        "image_features = self.multi_modal_projector(selected_image_feature)",
        "image_features = self.multi_modal_projector(selected_image_feature.to(self.multi_modal_projector.linear_1.weight.dtype))")
    if _p != _src:
        _llava_path.write_text(_p)
print("✅ modeling_llava.py patched")

# Patch evaluate_util.py to accept do_sample + temperature from cfg (idempotent)
eval_py = FIUBENCH_DIR / "evaluate_util.py"
_eval_src = eval_py.read_text()
if '_do_sample = getattr' not in _eval_src:
    _eval_src = _eval_src.replace(
        "    #now generate\n    if aspect_ratio_ids is not None:",
        "    _do_sample = getattr(cfg.generation, 'do_sample', False)\n    _temperature = getattr(cfg.generation, 'temperature', 1.0)\n    #now generate\n    if aspect_ratio_ids is not None:"
    )
    _eval_src = _eval_src.replace(
        "max_new_tokens=cfg.generation.max_new_tokens, do_sample=False, use_cache=True",
        "max_new_tokens=cfg.generation.max_new_tokens, do_sample=_do_sample, temperature=_temperature, use_cache=True"
    )
    eval_py.write_text(_eval_src)
    print("✅ Patched evaluate_util.py: do_sample + temperature from cfg")
else:
    print("✅ evaluate_util.py already patched")

# Patch eval.yaml to add do_sample and temperature fields (idempotent)
eval_yaml = FIUBENCH_DIR / "config/eval.yaml"
_yaml_src = eval_yaml.read_text()
if 'do_sample' not in _yaml_src:
    _yaml_src = _yaml_src.replace(
        "generation:\n  max_length: 256\n  max_new_tokens: 50",
        "generation:\n  max_length: 256\n  max_new_tokens: 50\n  temperature: 1.0\n  do_sample: false"
    )
    eval_yaml.write_text(_yaml_src)
    print("✅ Patched eval.yaml: added temperature + do_sample fields")
else:
    print("✅ eval.yaml already has do_sample field")

# Ensure SFHQ images exist (re-download if session reset wiped /content)
sfhq_dir     = Path(PROJECT_ROOT) / "data/datasets/datasets--gray311--FIUBench/fiubench_extracted/SFHQ"
sfhq_symlink = FIUBENCH_DIR / "dataset/SFHQ"

existing = list(sfhq_dir.glob("*.jpg")) if sfhq_dir.exists() else []
if len(existing) >= 400:
    print(f"✅ SFHQ images present: {len(existing)}")
else:
    print(f"⬇  SFHQ missing ({len(existing)}) — re-downloading...")
    from huggingface_hub import hf_hub_download
    sfhq_dir.mkdir(parents=True, exist_ok=True)
    zip_path = hf_hub_download(repo_id="gray311/FIUBench", filename="SFHQ.zip", repo_type="dataset")
    extract_dir = sfhq_dir.parent / "sfhq_extracted"
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(extract_dir)
    for src in extract_dir.rglob("*.jpg"):
        dst = sfhq_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
    print(f"✅ SFHQ ready: {len(list(sfhq_dir.glob('*.jpg')))} images")

if sfhq_symlink.is_symlink() and not sfhq_symlink.exists():
    sfhq_symlink.unlink()
if not sfhq_symlink.exists():
    sfhq_symlink.parent.mkdir(parents=True, exist_ok=True)
    sfhq_symlink.symlink_to(sfhq_dir)
    print("✅ SFHQ symlink created")
else:
    print("✅ SFHQ symlink OK")

os.chdir(str(FIUBENCH_DIR))

# ── Generation configs to test ────────────────────────────────────────────────
CONFIGS = [
    ("greedy",   False, None),
    ("temp=0.1", True,  0.1),
    ("temp=0.3", True,  0.3),
    ("temp=0.7", True,  0.7),
    ("temp=1.0", True,  1.0),
]

results = {}

for label, do_sample, temperature in CONFIGS:
    save_dir = f"../results/temp_sweep/{label.replace('=','_')}"
    Path(save_dir).mkdir(parents=True, exist_ok=True)

    cmd = [
        'python', 'evaluate_util.py', '--config-name', 'eval',
        f'model_path={MODEL_PATH}',
        'LoRA.r=0',
        f'save_dir={save_dir}',
        'split_list=[retain5]',
        'eval_task=[eval_retain_log]',
        'robust_eval=[[rouge]]',
        'batch_size=1',
        'perturb_batch_size=1',
        'generation.max_new_tokens=50',
        f'generation.do_sample={str(do_sample).lower()}',
        'overwrite=true',
        f'hydra.run.dir=/tmp/hydra_sweep_{label.replace("=","_")}',
    ]
    if temperature is not None:
        cmd.append(f'generation.temperature={temperature}')

    print(f"\n{'─'*50}")
    print(f"Running: {label}")
    print(f"{'─'*50}")

    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        print(f"❌ FAILED (exit {proc.returncode})")
        results[label] = None
        continue

    result_file = Path(save_dir) / 'retain5_eval_retain_log.json'
    if not result_file.exists():
        print(f"❌ Result file not found")
        results[label] = None
        continue

    data   = json.load(open(result_file))
    rouge  = data.get('rougeL_recall', {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = np.mean(scores) * 100
    results[label] = mean_rouge
    print(f"✅ ROUGE-L recall: {mean_rouge:.1f}%")

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'='*50}")
print(f"{'Config':<14} {'ROUGE-L':>10}  {'vs greedy':>10}")
print(f"{'─'*50}")
baseline = results.get("greedy")
for label, score in results.items():
    if score is None:
        print(f"{label:<14} {'FAILED':>10}")
    else:
        delta = f"{score - baseline:+.1f}pp" if baseline is not None and label != "greedy" else "—"
        print(f"{label:<14} {score:>9.1f}%  {delta:>10}")
print(f"{'─'*50}")
print(f"{'Paper target':<14} {'93.3':>10}%")
print(f"{'='*50}")
if baseline is not None:
    valid = {l: s for l, s in results.items() if s is not None}
    best_label = max(valid, key=lambda l: valid[l])
    best_score = valid[best_label]
    if best_score > baseline + 1.0:
        print(f"\n→ Temperature matters: best={best_label} ({best_score:.1f}%), greedy={baseline:.1f}%")
    else:
        print(f"\n→ Temperature has minimal effect — training gap is the dominant factor")


## Stage 1 Decoding Temperature Analysis
*Target Split: `retain5` (400 samples)*

| Configuration | ROUGE-L | vs greedy |
|:---|:---|:---|
| **greedy** | **76.6%** | **—** |
| temp=0.1 | 76.6% | -0.0pp |
| temp=0.3 | 75.9% | -0.7pp |
| temp=0.7 | 73.2% | -3.4pp |
| temp=1.0 | 69.5% | -7.1pp |
| **Paper target** | **93.3%** | |

### So temperature close to do_sample=False or 0.1 is best for high Rouge-L score 

## GPT-Eval Wrapper (GPT-4o-mini)

Same prompt as `FIUBench/gpt_eval.py`. Scores each prediction 0–1 for factual accuracy.

In [30]:
import os, re, time
from openai import OpenAI

_GPT_PROMPT = """You are an intelligent chatbot designed for evaluating the factual accuracy of generative outputs for question-answer pairs about fictitious entities.
Your task is to compare the predicted answer with the correct answer and determine if they are factually consistent. Here's how you can accomplish the task:
1. Focus on the meaningful match between the predicted answer and the correct answer.
2. Consider synonyms or paraphrases as valid matches.
3. Evaluate the correctness of the prediction compared to the answer.
4. Please do not consider the difference in sentence style between the correct answer and the predicted answer, but only judge whether the predicted answer makes sense based on factual accuracy.
5. If there is something in the predicted answer that is not in the correct answer, then it is considered to be hallucination.

The score should range from 0 to 1. A larger score means a better answer. The score should be a float number with 2 decimal places.
Please first output a single line containing only one value indicating the scores for the predicted answer.
In the subsequent line, please provide some key words of the question and correct answers.
In the subsequent line, please provide a comprehensive explanation of your evaluation.

Question: {question}
Correct Answer: {answer}
Prediction: {prediction}

Outputs (include score, key words, explanation):"""

def _parse_gpt_score(text: str) -> float:
    line = text.strip().split("\n")[0].strip()
    line = re.sub(r"\*\*", "", line)
    if ":" in line:
        line = line.split(":")[-1].strip()
    try:
        return float(line)
    except ValueError:
        nums = re.findall(r"\d+\.\d+", line)
        return float(nums[0]) if nums else float("nan")

def _extract_question(inp: str) -> str:
    """Extract clean question text from evaluate_util.py input string (handles Phi-3 and LLaVA formats)."""
    if "USER:" in inp:
        q = inp[inp.find("USER:"):].replace("USER:", "").replace("<image>", "").strip()
    elif "<|assistant|>" in inp:
        q = inp.split("<|assistant|>")[0]
        for tok in ["<s>", "<image>", "<|user|>", "<|end|>"]:
            q = q.replace(tok, "")
        q = q.strip()
    else:
        q = inp.split("ASSISTANT")[0].replace("<image>", "").strip()
    return q

def gpt_eval_results(result_json: dict, api_key: str, n: int = 50,
                     model: str = "gpt-4o-mini") -> float:
    """Run GPT-Eval on generated_text from an evaluate_util.py result JSON. Returns mean score."""
    if not api_key:
        print("  ⚠️  No OPENAI_API_KEY — GPT-Eval skipped")
        return float("nan")
    gen_texts = result_json.get("generated_text", {})
    if not gen_texts:
        print("  ⚠️  No generated_text in result")
        return float("nan")
    client = OpenAI(api_key=api_key)
    scores = []
    for idx, val in list(gen_texts.items())[:n]:
        inp, gen, gt, cat = val
        q = _extract_question(inp)
        prompt = _GPT_PROMPT.format(question=q, answer=gt, prediction=gen)
        for attempt in range(3):
            try:
                resp = client.chat.completions.create(
                    model=model,
                    messages=[{"role": "user", "content": prompt}],
                    max_tokens=100, temperature=0.0,
                )
                scores.append(_parse_gpt_score(resp.choices[0].message.content))
                break
            except Exception as e:
                if attempt == 2:
                    scores.append(float("nan"))
                else:
                    time.sleep(2 ** attempt)
    valid = [s for s in scores if s == s]
    mean = sum(valid) / len(valid) if valid else float("nan")
    print(f"  GPT-Eval: {len(valid)}/{len(scores)} scored, mean = {mean:.3f}")
    return mean

# Smoke tests
assert abs(_parse_gpt_score("0.85\nkeywords\nexplanation") - 0.85) < 1e-6
assert _extract_question('<s> \nWhat is the blood type?<|end|><|assistant|>') == 'What is the blood type?'
assert _extract_question('USER: <image>\nWhat is the name?ASSISTANT:') == 'What is the name?ASSISTANT:'
print("✅ GPT-Eval wrapper ready")

✅ GPT-Eval wrapper ready


## Table 1 — Pretrained vs Finetuned (Rouge-L + GPT-Eval)

Loads already-saved eval JSON files (no re-inference needed) and runs GPT-Eval on the generated outputs.

In [32]:
import json, numpy as np
from pathlib import Path

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")  # set this before running
GPT_N = 50  # samples to score per model (full=400, use 50 for cost control)

EVALS = {
    "Pretrained":  Path("/content/FIUBench_Reproducing/results/baseline_eval_retain5/retain5_eval_retain_log.json"),
    "Finetuned":   Path("/content/FIUBench_Reproducing/results/stage1_eval_retain5/retain5_eval_retain_log.json"),
}

results = {}
for label, path in EVALS.items():
    print(f"\n── {label} ──────────────────────────────────")
    if not path.exists():
        print(f"  ❌ Result file not found: {path}")
        results[label] = {"rouge_l": float("nan"), "gpt": float("nan")}
        continue

    data   = json.load(open(path))
    rouge  = data.get("rougeL_recall", {})
    scores = [float(v) for v in rouge.values() if v is not None]
    mean_rouge = float(np.mean(scores)) * 100 if scores else float("nan")
    print(f"  Rouge-L (retain5): {mean_rouge:.1f}%")

    gpt_score = gpt_eval_results(data, api_key=OPENAI_API_KEY, n=GPT_N)

    results[label] = {"rouge_l": mean_rouge, "gpt": gpt_score}

# ── Table 1 ───────────────────────────────────────────────────────────────────
print("\n")
print("Table 1: Model performance before and after Stage 1 fine-tuning")
print(f"{'─'*65}")
print(f"{'Model':<28} {'Pretrained':>16} {'':>4} {'Finetuned':>16}")
print(f"{'':28} {'Rouge-L':>8} {'GPT':>8} {'':>4} {'Rouge-L':>8} {'GPT':>8}")
print(f"{'─'*65}")

pre  = results.get("Pretrained",  {})
fine = results.get("Finetuned",   {})

def fmt_rouge(v):
    return f"{v:.1f}" if v == v else "—"
def fmt_gpt(v):
    return f"{v:.2f}" if v == v else "—"
def arrow(pre_v, fine_v, higher_is_better=True):
    if pre_v != pre_v or fine_v != fine_v:
        return ""
    return "↑" if (fine_v > pre_v) == higher_is_better else "↓"

r_arrow = arrow(pre.get("rouge_l", float("nan")), fine.get("rouge_l", float("nan")))
g_arrow = arrow(pre.get("gpt",     float("nan")), fine.get("gpt",     float("nan")))

print(
    f"{'LLaVA-Phi-Mini-3B':<28}"
    f" {fmt_rouge(pre.get('rouge_l', float('nan'))):>8}"
    f" {fmt_gpt(pre.get('gpt',     float('nan'))):>8}"
    f" {'':>4}"
    f" {fmt_rouge(fine.get('rouge_l', float('nan'))) + r_arrow:>8}"
    f" {fmt_gpt(fine.get('gpt',     float('nan'))) + g_arrow:>8}"
)
print(f"{'─'*65}")
print(f"Paper (LLaVA-Phi-Mini-3B)   {'53.6':>8} {'0.07':>8} {'':>4} {'93.3↑':>8} {'85.8↑':>8}")
print(f"{'─'*65}")


── Pretrained ──────────────────────────────────
  Rouge-L (retain5): 58.7%
  GPT-Eval: 50/50 scored, mean = 0.089

── Finetuned ──────────────────────────────────
  Rouge-L (retain5): 75.5%
  GPT-Eval: 50/50 scored, mean = 0.190


Table 1: Model performance before and after Stage 1 fine-tuning
─────────────────────────────────────────────────────────────────
Model                              Pretrained             Finetuned
                              Rouge-L      GPT       Rouge-L      GPT
─────────────────────────────────────────────────────────────────
LLaVA-Phi-Mini-3B                58.7     0.09         75.5↑    0.19↑
─────────────────────────────────────────────────────────────────
Paper (LLaVA-Phi-Mini-3B)       53.6     0.07         93.3↑    85.8↑
─────────────────────────────────────────────────────────────────


# !!!Maybe try 1 batchsize and 4 accumulation steps